BB84: Quantum Key Distribution  
This is a walk through of a simple implementation of the BB84 protocol for QKD. 
This note book walks through my implementation of the BB84 QKD protocol in current Qiskit.
It contains Alice (sender), Bob (receiver) and Eve (eavesdropper).
The evesdropper is an incerpet-resend and created a detecable disturance via the quanutm bit error rate (QBER)
My dissertion covered the theory of BB84 and E91 finishing on DI-QKD. The goal here was to further my understandiung by amking the theory executable  - every gate justified, every statistic checked against prediction.
The full implementation lives in protocols/, attacks/ and analysis/; this notebook imports it and demonstrates it stage by stage.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

from protocols.bb84 import encode_qubit, measure_qubit, sift, qber, run_bb84

sim = AerSimulator() # create a simulator instance
rng = np.random.default_rng(42) # create a random number generator instance with a fixed seed for reproducibility

One qubit, two orthnormal bases
BB84 relies geavily on the one linear -algerbra fact that the Hadamard fate is a change of basis. It maps the compuational Z basis onto the diagonal X basis,  H|0⟩ = |+⟩ and H|1⟩ = |−⟩. Because H² = I, applying it a second time maps them back.
The gate itself is deterministic - |0⟩ becomes exactly |+⟩ every single time. The randomness only enters at measurement.
The Born rule gives P(outcome) = |amplitude|², so measuring |+⟩ in the Z basis is a fair coin: |⟨0|+⟩|^2 = 1/2.
The next cell shows both cases side by side: the same X-basis state measured in the wrong basis (random) and in the right basis (certain).

In [ ]:
# Same X-basis state, measured in the wrong basis vs the right basis
qc_wrong = QuantumCircuit(1, 1); qc_wrong.h(0); qc_wrong.measure(0, 0) # creates a quanutm ciructe that prepares a qubit in the X-basis and measures it in the Z-basis
qc_right = QuantumCircuit(1, 1); qc_right.h(0); qc_right.h(0); qc_right.measure(0, 0) # creates a quantum circuit that prepares a qubit in the X-basis and measures it in the X-basis

for label, qc in [("Z-measure (orthogonal basis)", qc_wrong), ("X-measure (matching basis)", qc_right)]:
    counts = sim.run(transpile(qc, sim), shots=1000).result().get_counts()
    print(f"{label}: {counts}")

When measuring using the orthnoal basis measurment splits roughly 50/50. The matching basis retunrs the encoded value 100% of the time, because Bob's Hadamard undoes Alice's (H² = I) and the state hitting the detector is exactly |1⟩ - the Born rule then gives probability 1, not a coin flip.
This asymmetry is the entire security mechanism of BB84: deterministic in the matching basis, uniformly random in the mismatched one. Anyone who measures without knowing the basis destroys information and leaves statistical fingerprints. The rest of the notebook is about making those fingerprints visible.


The protocol on a clean channel 